In [1]:
# Cell 1: Imports & Setup
from pathlib import Path
import json
import numpy as np
import pandas as pd
import polars as pl
import gc

data_base_dir = Path('data/regionwise_yearwise_samples')
output_dir = Path('data/feature_selection_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

# TARGET_REGION = 'DES'
# TARGET_REGIONS = ['IGP', 'WGR', 'ICR']
TARGET_REGIONS = ['IGP', 'WGR', 'ICR','DES','NEI', 'PEN','HIM']
# REGION_DIR = data_base_dir / TARGET_REGION

cols_to_drop = [
    'class', 'system:index', 'area_m2', 'area_proportion', 'usable_count',
    '.geo', 'q1_count', 'q2_count', 'q3_count', 'q4_count', 'quarters_present',
    'geometry', 'ndvi_q1_median', 'ndvi_q2_median', 'ndvi_q3_median',
    'ndvi_q4_median', 'tir_count', 'tir_mad', 'tir_median', 'snow_count',
    'year'
]

MIN_USABLE_LOOKS = 5    # Minimum clear looks in a phenological year
MIN_QUARTERS = 3        # Minimum quarters observed out of 4 (q1..q4)

# Define the predicate for believing a single year's observation at a pixel
pixel_quality_condition = (
    (pl.col("usable_count") >= MIN_USABLE_LOOKS) &
    (pl.col("quarters_present") >= MIN_QUARTERS)
)

old_codes = [5, 3, 4, 12, 11, 66, 19, 36, 9, 41, 24, 23, 75, 30, 25, 33, 76, 34, 61]
new_codes = [5, 3, 4, 12, 11, 66, 19, 14, 14, 14, 22, 22, 22, 22, 22, 33, 76, 34, 61]
class_remap_dict = dict(zip(old_codes, new_codes))

In [2]:
# Cell 2: Compute Correlation & Drop Collinear Features (Iterative)
for target_region in TARGET_REGIONS:
    print(f"\n{'='*50}")
    print(f"Processing region: {target_region}")
    print(f"{'='*50}")
    
    region_dir = data_base_dir / target_region
    
    # --- 1. Sample 50k Rows for Fast Correlation ---
    lazy_ds = pl.scan_parquet(region_dir / "**/*.parquet")
    schema = lazy_ds.collect_schema()
    feature_cols = [c for c in schema.names() if c not in cols_to_drop]

    # Collect the full dataset into memory first
    df_collected = (
        lazy_ds
        .filter(pixel_quality_condition)
        .with_columns(
            pl.col('class').replace_strict(class_remap_dict, default=pl.col('class')).cast(pl.Int32)
        )
        .select(feature_cols + ['class'])
        .collect()
    )
    
    region_total_rows = df_collected.height
    print(f"[{target_region}] Total available rows: {region_total_rows:,}")

    # Dynamically sample up to 50k rows using the region's actual size
    df_sample = df_collected.sample(n=min(50000, region_total_rows), seed=42)

    # Free the large unsampled dataframe from memory
    del df_collected
    gc.collect()

    X_sample = df_sample.select(feature_cols).to_numpy().astype(np.float32)
    y_sample = df_sample['class'].to_numpy().astype(np.float32)

    del df_sample
    gc.collect()

    print(f"[{target_region}] Correlation sample ready: {X_sample.shape[0]:,} rows across {len(feature_cols)} features.")
    
    # --- 2. Compute Pairwise & Target Correlation Matrices ---
    # Feature-to-Feature Pearson correlation
    corr_matrix = np.abs(np.corrcoef(X_sample, rowvar=False))
    corr_df = pd.DataFrame(corr_matrix, index=feature_cols, columns=feature_cols)

    # Vectorized Feature-to-Target correlation (|r(X, y)|)
    y_norm = (y_sample - np.mean(y_sample)) / np.std(y_sample)
    X_norm = (X_sample - np.mean(X_sample, axis=0)) / np.std(X_sample, axis=0)
    target_corr = np.abs(np.mean(X_norm * y_norm[:, np.newaxis], axis=0))
    target_corr_series = pd.Series(target_corr, index=feature_cols)
    
    # --- 3. Drop Collinear Features (|r| > 0.80) & Export Artifact ---
    threshold = 0.80
    upper = corr_df.where(np.triu(np.ones(corr_df.shape), k=1).astype(bool))

    to_drop = set()
    for col in upper.columns:
        correlated_features = upper.index[upper[col] > threshold].tolist()
        if correlated_features:
            for feat in correlated_features:
                if target_corr_series[feat] < target_corr_series[col]:
                    to_drop.add(feat)
                else:
                    to_drop.add(col)

    uncorrelated_features = [col for col in feature_cols if col not in to_drop]

    output_artifact = {
        "region": target_region,
        "corr_threshold": threshold,
        "total_raw_features": len(feature_cols),
        "uncorrelated_features": uncorrelated_features,
        "dropped_features": sorted(list(to_drop))
    }

    output_path = output_dir / f"{target_region}_uncorrelated_features.json"
    with open(output_path, "w") as f:
        json.dump(output_artifact, f, indent=2)

    print(f"[{target_region}] Initial raw features: {len(feature_cols)}")
    print(f"[{target_region}] Collinear features dropped (|r| > {threshold}): {len(to_drop)}")
    print(f"[{target_region}] Retained uncorrelated features: {len(uncorrelated_features)}")
    print(f"[{target_region}] Artifact saved to -> {output_path}")
    
    # Clear arrays and matrices to free memory before next region
    del X_sample, y_sample, corr_matrix, corr_df, y_norm, X_norm, target_corr, target_corr_series, upper, output_artifact
    gc.collect()


Processing region: IGP
[IGP] Total available rows: 2,817,941
[IGP] Correlation sample ready: 50,000 rows across 104 features.


/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_39788/2995065310.py:50: RuntimeWarning: invalid value encountered in divide
  X_norm = (X_sample - np.mean(X_sample, axis=0)) / np.std(X_sample, axis=0)


[IGP] Initial raw features: 104
[IGP] Collinear features dropped (|r| > 0.8): 68
[IGP] Retained uncorrelated features: 36
[IGP] Artifact saved to -> data/feature_selection_artifacts/IGP_uncorrelated_features.json

Processing region: WGR
[WGR] Total available rows: 1,763,694
[WGR] Correlation sample ready: 50,000 rows across 106 features.


/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_39788/2995065310.py:50: RuntimeWarning: invalid value encountered in divide
  X_norm = (X_sample - np.mean(X_sample, axis=0)) / np.std(X_sample, axis=0)


[WGR] Initial raw features: 106
[WGR] Collinear features dropped (|r| > 0.8): 70
[WGR] Retained uncorrelated features: 36
[WGR] Artifact saved to -> data/feature_selection_artifacts/WGR_uncorrelated_features.json

Processing region: ICR
[ICR] Total available rows: 2,704,756
[ICR] Correlation sample ready: 50,000 rows across 106 features.
[ICR] Initial raw features: 106
[ICR] Collinear features dropped (|r| > 0.8): 68
[ICR] Retained uncorrelated features: 38
[ICR] Artifact saved to -> data/feature_selection_artifacts/ICR_uncorrelated_features.json


...

/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_39788/2995065310.py:50: RuntimeWarning: invalid value encountered in divide
  X_norm = (X_sample - np.mean(X_sample, axis=0)) / np.std(X_sample, axis=0)


[NEI] Initial raw features: 104
[NEI] Collinear features dropped (|r| > 0.8): 68
[NEI] Retained uncorrelated features: 36
[NEI] Artifact saved to -> data/feature_selection_artifacts/NEI_uncorrelated_features.json

Processing region: PEN
[PEN] Total available rows: 1,786,187
[PEN] Correlation sample ready: 50,000 rows across 104 features.


/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_39788/2995065310.py:50: RuntimeWarning: invalid value encountered in divide
  X_norm = (X_sample - np.mean(X_sample, axis=0)) / np.std(X_sample, axis=0)


[PEN] Initial raw features: 104
[PEN] Collinear features dropped (|r| > 0.8): 67
[PEN] Retained uncorrelated features: 37
[PEN] Artifact saved to -> data/feature_selection_artifacts/PEN_uncorrelated_features.json

Processing region: HIM
[HIM] Total available rows: 1,448,209
[HIM] Correlation sample ready: 50,000 rows across 104 features.
[HIM] Initial raw features: 104
[HIM] Collinear features dropped (|r| > 0.8): 73
[HIM] Retained uncorrelated features: 31
[HIM] Artifact saved to -> data/feature_selection_artifacts/HIM_uncorrelated_features.json


/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/nirzaree/Code/GitClones/IOLN/.IOLN/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_39788/2995065310.py:50: RuntimeWarning: invalid value encountered in divide
  X_norm = (X_sample - np.mean(X_sample, axis=0)) / np.std(X_sample, axis=0)
